In [0]:
%run ./01_logging_utils.py

In [0]:

from datetime import datetime
from pyspark.sql import functions as F


def run_backfill(start_date, end_date):
    """
    Reprocess Bronze records for an inclusive date range
    and merge the cleaned results into Silver.

    Dates must use YYYY-MM-DD format.
    """
    run_id = new_batch_id("backfill")
    start_time = datetime.now()

    try:
        start = F.to_date(F.lit(start_date))
        end = F.to_date(F.lit(end_date))

        bronze_df = spark.table(T_BRONZE)

        # Convert millisecond timestamps to timestamps
        transformed_df = (
            bronze_df
            .withColumn(
                "event_ts",
                F.to_timestamp(
                    F.from_unixtime(F.col("ts_ms") / 1000.0)
                )
            )
            .withColumn("event_date", F.to_date("event_ts"))
            .withColumn("event_hour", F.hour("event_ts"))
        )

        # Filter to the requested inclusive date range
        filtered_df = transformed_df.filter(
            (F.col("event_date") >= start)
            & (F.col("event_date") <= end)
        )

        # Keep only valid records
        valid_df = filtered_df.filter(
            F.col("coin_id").isNotNull()
            & F.col("event_ts").isNotNull()
            & F.col("price").isNotNull()
            & (F.col("price") >= 0)
            & (
                F.col("market_cap").isNull()
                | (F.col("market_cap") >= 0)
            )
            & (
                F.col("total_volume").isNull()
                | (F.col("total_volume") >= 0)
            )
        )

        # Match the Silver table schema and remove duplicates
        valid_df = (
            valid_df
            .select(
                "coin_id",
                "event_ts",
                "price",
                "market_cap",
                "total_volume",
                "event_date",
                "event_hour",
                "batch_id",
                "load_timestamp"
            )
            .dropDuplicates(["coin_id", "event_ts"])
        )

        rows_processed = valid_df.count()

        # Merge into Silver using the coin and timestamp key
        valid_df.createOrReplaceTempView("backfill_source")

        spark.sql(f"""
            MERGE INTO {T_SILVER} AS target
            USING backfill_source AS source
            ON target.coin_id = source.coin_id
               AND target.event_ts = source.event_ts

            WHEN MATCHED AND NOT (
                target.price <=> source.price
                AND target.market_cap <=> source.market_cap
                AND target.total_volume <=> source.total_volume
                AND target.event_date <=> source.event_date
                AND target.event_hour <=> source.event_hour
            )
            THEN UPDATE SET *

            WHEN NOT MATCHED THEN INSERT *
        """)

        inserted, updated = merge_metrics(T_SILVER)

        log_run(
            run_id,
            "backfill",
            f"start_date={start_date}, end_date={end_date}",
            start_time,
            datetime.now(),
            "SUCCESS",
            inserted,
            updated
        )

        print("Backfill completed successfully.")
        print(f"Date range: {start_date} to {end_date}")
        print(f"Valid rows processed: {rows_processed}")
        print(f"Rows inserted: {inserted}")
        print(f"Rows updated: {updated}")

    except Exception as e:
        log_run(
            run_id,
            "backfill",
            f"start_date={start_date}, end_date={end_date}",
            start_time,
            datetime.now(),
            "FAILED",
            err=str(e)[:500]
        )
        raise


In [0]:
display(spark.sql("""
    SELECT
        MIN(to_date(timestamp_seconds(ts_ms / 1000))) AS earliest_date,
        MAX(to_date(timestamp_seconds(ts_ms / 1000))) AS latest_date
    FROM crypto.bronze.market_data_raw
"""))

In [0]:
run_backfill("2026-09-04", "2026-10-04")